# Grounded Segmentation

Inspect a failed box. Can the mask stage repair the wrong object selection?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/facebook/sam-vit-base

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os, random, requests, tarfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    SamModel, SamProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
GDINO_ID = "IDEA-Research/grounding-dino-tiny"
print(f"📦 Grounding DINO: {GDINO_ID}")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(device)
gdino_model.eval()
gdino_params = sum(p.numel() for p in gdino_model.parameters())
print(f"   ✅ Done! ({gdino_params/1e6:.1f}M params)")

In [ ]:
SAM_ID = "facebook/sam-vit-base"
print(f"📦 SAM: {SAM_ID}")
sam_processor = SamProcessor.from_pretrained(SAM_ID)
sam_model = SamModel.from_pretrained(SAM_ID).to(device)
sam_model.eval()
sam_params = sum(p.numel() for p in sam_model.parameters())
print(f"   ✅ Done! ({sam_params/1e6:.1f}M params)")
print(f"\n📊 Total: {(gdino_params + sam_params)/1e6:.1f}M params")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
import hashlib
_detection_cache = {}
_sam_embedding_cache = {}

@torch.no_grad()
def detect_objects(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """
    Use Grounding DINO to detect objects matching the text in an image.

    Args:
        image: PIL Image
        text_labels: list of str (e.g., ["dog", "cat"])
    Returns:
        boxes (N,4), scores (N,), labels list[str]
    """
    key = (hashlib.sha256(image.tobytes()).hexdigest(), image.size, tuple(text_labels))
    if key not in _detection_cache:
        inputs = gdino_processor(images=image, text=[text_labels], return_tensors="pt").to(device)
        _detection_cache[key] = gdino_model(**inputs)
    outputs = _detection_cache[key]
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, threshold=box_threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)]
    )
    r = results[0]
    return r['boxes'].cpu().numpy(), r['scores'].cpu().numpy(), r['text_labels']

print("✅ detect_objects defined")

In [ ]:
def draw_detections(image, boxes, scores, labels, title='', ax=None):
    if ax is None: fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image)
    unique = list(set(labels))
    cmap = plt.cm.Set2
    cdict = {l: cmap(i / max(len(unique), 1)) for i, l in enumerate(unique)}
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        c = cdict[label]
        rect = mpatches.FancyBboxPatch((x1,y1), x2-x1, y2-y1, linewidth=3,
            edgecolor=c, facecolor='none', boxstyle='round,pad=0')
        ax.add_patch(rect)
        ax.text(x1, y1-8, f'{label}: {score:.2f}', fontsize=11, fontweight='bold',
                color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=c, edgecolor='none', alpha=0.85))
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')
    return ax

print("✅ draw_detections defined")

In [ ]:
@torch.no_grad()
def cached_sam_inputs(image, **prompts):
    """Reuse the frozen image encoder when only a prompt changes."""
    key = (hashlib.sha256(image.tobytes()).hexdigest(), image.size)
    inputs = sam_processor(images=image, return_tensors="pt", **prompts).to(device)
    pixels = inputs.pop("pixel_values")
    if key not in _sam_embedding_cache:
        _sam_embedding_cache[key] = sam_model.get_image_embeddings(pixels)
    inputs["image_embeddings"] = _sam_embedding_cache[key]
    return inputs

@torch.no_grad()
def segment_with_boxes(image, boxes):
    """
    Pass bounding boxes to SAM as prompts to generate segmentation masks.
    Process boxes one at a time to ensure shape stability.

    Args:
        image: PIL Image
        boxes: numpy array (N, 4) [x1, y1, x2, y2]
    Returns:
        masks: list of (H, W) bool numpy arrays
        iou_scores: list of float
    """
    if len(boxes) == 0:
        return [], []

    best_masks = []
    best_scores = []

    for box in boxes:
        # SAM processor expects: input_boxes = [[[x1, y1, x2, y2]]]
        input_boxes = [[box.tolist()]]
        inputs = cached_sam_inputs(image, input_boxes=input_boxes)

        outputs = sam_model(**inputs)

        # post_process_masks: restore to original image size
        masks_list = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu()
        )
        iou = outputs.iou_scores.cpu().numpy()  # (1, 1, 3)

        # masks_list[0] shape: (1, 3, H, W) → squeeze → (3, H, W)
        mask_3 = masks_list[0].squeeze(0)   # (3, H, W)
        score_3 = iou[0, 0]                  # (3,)

        best_idx = score_3.argmax()
        best_masks.append(mask_3[best_idx].numpy().astype(bool))
        best_scores.append(float(score_3[best_idx]))

    return best_masks, best_scores


@torch.no_grad()
def segment_with_points(image, points, point_labels=None):
    """Pass point prompts to SAM to generate a segmentation mask."""
    if point_labels is None:
        point_labels = [1] * len(points)

    inputs = cached_sam_inputs(image, input_points=[[points]], input_labels=[[point_labels]])

    outputs = sam_model(**inputs)
    masks_list = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    iou = outputs.iou_scores.cpu().numpy()
    mask_3 = masks_list[0].squeeze(0)  # (3, H, W)
    score_3 = iou[0, 0]                # (3,)
    best_idx = score_3.argmax()
    return mask_3[best_idx].numpy().astype(bool), float(score_3[best_idx])

print("✅ SAM segmentation functions defined")

In [ ]:
def grounded_sam(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """Grounded SAM: Detection → Segmentation"""
    boxes, scores, labels = detect_objects(image, text_labels, box_threshold, text_threshold)
    if len(boxes) == 0:
        return boxes, scores, labels, [], []
    masks, ious = segment_with_boxes(image, boxes)
    return boxes, scores, labels, masks, ious

print("✅ grounded_sam pipeline defined")

In [ ]:
def visualize_grounded_sam(image, boxes, scores, labels, masks, title=''):
    """Original / Detection / Mask / Segmented — 4-panel visualization"""
    from scipy import ndimage
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    img_arr = np.array(image)

    axes[0].imshow(image); axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')
    draw_detections(image, boxes, scores, labels,
                   title=f'Grounding DINO ({len(boxes)} boxes)', ax=axes[1])

    axes[2].imshow(image)
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
    for i, m in enumerate(masks):
        c = colors[i % len(colors)]
        rgba = np.zeros((*m.shape, 4)); rgba[m] = [c[0], c[1], c[2], 0.5]
        axes[2].imshow(rgba)
        edge = ndimage.binary_dilation(m, iterations=2) ^ m
        edge_rgba = np.zeros((*m.shape, 4)); edge_rgba[edge] = [c[0], c[1], c[2], 1.0]
        axes[2].imshow(edge_rgba)
    axes[2].set_title(f'SAM ({len(masks)} masks)', fontweight='bold'); axes[2].axis('off')

    if masks:
        combined = np.any(np.stack(masks), axis=0)
        seg = img_arr.copy(); seg[~combined] = 255
        axes[3].imshow(seg)
    else:
        axes[3].imshow(image)
    axes[3].set_title('Segmented', fontweight='bold'); axes[3].axis('off')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

print("✅ visualize_grounded_sam defined")

## Demo code

Inspect a failed box. Can the mask stage repair the wrong object selection?

In [ ]:
image = Image.open(image_paths[0]).convert("RGB")
boxes, scores, labels, masks, quality = grounded_sam(image, ["dog"], box_threshold=0.30)
visualize_grounded_sam(image, boxes, scores, labels, masks)
print("Detected boxes:", len(boxes), "Masks:", len(masks))

In [ ]:
print("🚀 Running the Grounded SAM pipeline\n")

demo_indices = [0, 3, NUM_IMAGES_PER_CLASS, NUM_IMAGES_PER_CLASS + 3]
for idx in demo_indices:
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx]
    boxes, scores, labels, masks, ious = grounded_sam(img, [gt], box_threshold=0.25)
    visualize_grounded_sam(img, boxes, scores, labels, masks,
        title=f'Prompt: "{gt}" | GT: {gt.upper()}')

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')

creative_prompts = [
    (["dog"],                   '"dog"'),
    (["head", "body"],          '"head", "body"'),
    (["furry animal"],          '"furry animal"'),
    (["pet", "background"],     '"pet", "background"'),
]

fig, axes = plt.subplots(2, len(creative_prompts), figsize=(6*len(creative_prompts), 12))
fig.suptitle('🧪 Open-Vocabulary Grounded SAM', fontsize=16, fontweight='bold')

for col, (text_labels, name) in enumerate(creative_prompts):
    boxes, scores, labels, masks, ious = grounded_sam(sample_img, text_labels, box_threshold=0.2)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'Prompt: {name}\n{len(boxes)} detections', ax=axes[0, col])
    axes[1, col].imshow(sample_img)
    if masks:
        colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
        for i, m in enumerate(masks):
            rgba = np.zeros((*m.shape, 4))
            rgba[m] = [colors[i][0], colors[i][1], colors[i][2], 0.5]
            axes[1, col].imshow(rgba)
    axes[1, col].set_title(f'{len(masks)} masks'); axes[1, col].axis('off')

plt.tight_layout(); plt.show()

In [ ]:
n_demo = 6
demo_paths = random.sample(image_paths, n_demo)
fig, axes = plt.subplots(3, n_demo, figsize=(4*n_demo, 12))
fig.suptitle('✂️ Grounded SAM — Automatic Background Removal', fontsize=16, fontweight='bold')

for col, path in enumerate(demo_paths):
    img = Image.open(path).convert('RGB')
    gt = 'dog' if 'dog' in path else 'cat'
    img_arr = np.array(img)
    boxes, scores, labels, masks, ious = grounded_sam(img, [gt], box_threshold=0.25)

    axes[0, col].imshow(img); axes[0, col].set_title(gt.upper()); axes[0, col].axis('off')

    axes[1, col].imshow(img)
    if masks:
        all_m = np.any(np.stack(masks), axis=0)
        rgba = np.zeros((*all_m.shape, 4)); rgba[all_m] = [0, 0.8, 0.4, 0.4]
        axes[1, col].imshow(rgba)
    axes[1, col].set_title('Mask'); axes[1, col].axis('off')

    if masks:
        all_m = np.any(np.stack(masks), axis=0)
        checker = np.zeros((*img_arr.shape[:2], 3), dtype=np.uint8)
        blk = 20
        for r in range(0, checker.shape[0], blk):
            for c in range(0, checker.shape[1], blk):
                v = 200 if (r//blk + c//blk) % 2 == 0 else 255
                checker[r:r+blk, c:c+blk] = v
        alpha = all_m[:,:,np.newaxis].astype(float)
        result = (checker * (1 - alpha) + img_arr * alpha).astype(np.uint8)
        axes[2, col].imshow(result)
    else:
        axes[2, col].imshow(img)
    axes[2, col].set_title('BG Removed'); axes[2, col].axis('off')

plt.tight_layout(); plt.show()

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')
thresholds = [0.1, 0.2, 0.3, 0.5, 0.7]

fig, axes = plt.subplots(2, len(thresholds), figsize=(5*len(thresholds), 10))
fig.suptitle('⚙️ Effect of changing the box threshold', fontsize=16, fontweight='bold')

for col, thr in enumerate(thresholds):
    boxes, scores, labels, masks, ious = grounded_sam(sample_img, ["dog", "cat"], box_threshold=thr)
    draw_detections(sample_img, boxes, scores, labels,
                   title=f'thr={thr} | {len(boxes)} boxes', ax=axes[0, col])
    axes[1, col].imshow(sample_img)
    if masks:
        colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))
        for i, m in enumerate(masks):
            rgba = np.zeros((*m.shape, 4))
            rgba[m] = [colors[i][0], colors[i][1], colors[i][2], 0.5]
            axes[1, col].imshow(rgba)
    axes[1, col].set_title(f'{len(masks)} masks'); axes[1, col].axis('off')

plt.tight_layout(); plt.show()

## Interpretation

The displayed masks come from actual Grounding DINO + SAM inference.

Compare the output with the evidence in the input. A generated answer can be wrong.